In [1]:
!pip install -q datasets pandas matplotlib seaborn wordcloud konlpy scipy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ast import literal_eval
from scipy.stats import chi2_contingency, f_oneway
import re

plt.rcParams['font.family'] = 'NanumGothic'  # 한글 폰트
!apt-get -qq install fonts-nanum > /dev/null
import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

from datasets import load_dataset
ds = load_dataset("nvidia/Nemotron-Personas-Korea", split="train")
df = ds.to_pandas()
print(df.shape)
df.head(3)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 20.1 MB/s eta 0:00:00


README.md:   0%|          | 0.00/36.0k [00:00<?, ?B/s]

data/train-00000-of-00009.parquet:   0%|          | 0.00/220M [00:00<?, ?B/s]

data/train-00001-of-00009.parquet:   0%|          | 0.00/220M [00:00<?, ?B/s]

data/train-00002-of-00009.parquet:   0%|          | 0.00/220M [00:00<?, ?B/s]

...

data/train-00007-of-00009.parquet:   0%|          | 0.00/220M [00:00<?, ?B/s]

data/train-00008-of-00009.parquet:   0%|          | 0.00/220M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

(1000000, 26)


,uuid,professional_persona,sports_persona,arts_persona,travel_persona,culinary_persona,family_persona,persona,cultural_background,skills_and_expertise,...,marital_status,military_status,family_type,housing_type,education_level,bachelors_field,occupation,district,province,country
0,03b4f36a18e6469386d0286dddd513c8,"전기태 씨는 광주 서구의 하역 현장에서 수십 년간 짐을 쌓아 올리며, 지렛대 원리를...","전기태 씨는 주말마다 무등산 자락을 느릿느릿 걸으며 땀을 흘리고, 내려오는 길에 단...",전기태 씨는 거실 소파에 깊숙이 파묻혀 텔레비전에서 나오는 옛날 가요 프로그램을 보...,전기태 씨는 아내와 함께 전국의 역사 유적지를 찾아다니며 옛 조상들의 발취를 느끼는...,전기태 씨는 일주일에 한 번 배달 짜장면과 탕수육을 시켜 먹는 날을 손꼽아 기다리며...,"전기태 씨는 전·월세 아파트에서 평생의 동반자인 아내와 단출하게 살아가며, 투박한 ...","전기태 씨는 광주 서구에서 평생 하역 일을 하며 살아온 70대 가장으로, 투박한 손...","광주 서구에서 평생을 보내며 투박하지만 정겨운 전라도 사투리가 몸에 배어 있고, 시...",수십 년간 하역 현장에서 다져진 감각으로 짐의 무게 중심을 한눈에 파악해 가장 효율...,...,배우자있음,비현역,배우자와 거주,아파트,초등학교,해당없음,하역 및 적재 관련 단순 종사원,광주-서구,광주,대한민국
1,73f75d42a3934626b0d9a4bff062715a,최은지 씨는 서초동 부동산 사무실에서 장부를 잡으며 복잡한 취득세나 양도세 계산을 ...,최은지 씨는 서초동 예술의전당 근처 산책로를 느릿하게 거닐며 동네 이웃들과 수다를 ...,"최은지 씨는 지역 문화센터의 서예 교실에서 붓을 잡으며 마음을 다스리지만, 쉬는 시...",최은지 씨는 가족들과 함께 경주 불국사의 다보탑이나 부여 낙화암의 절벽을 방문해 옛...,"최은지 씨는 일주일에 대여섯 번은 집 밖에서 식사를 해결하며, 나물 비빔밥이나 청국...",최은지 씨는 남편과 자녀들이 함께 사는 복작복작한 집안 분위기 속에서 가장 영향력 ...,최은지 씨는 서초구에서 부동산 회계 사무원으로 일하며 경제적 자립과 사교적인 삶을 ...,최은지는 서초구의 오래된 다세대 주택가에서 나고 자라며 체면과 질서를 중시하는 분위...,"복잡한 부동산 세금 계산이나 장부 정리를 빠르게 처리하며, 오랜 실무 경험을 바탕으...",...,배우자있음,비현역,배우자·자녀와 거주,다세대주택,4년제 대학교,자연과학·수학,회계 사무원,서울-서초구,서울,대한민국
2,89eca80b88284888ad94c84f56777680,"안상식 씨는 퇴직 후에도 목동 주민센터의 복잡한 서류 뭉치를 보며 흐뭇해하며, 동네...","안상식 씨는 정해진 시간에 집 근처 공원을 천천히 거닐며 나무의 상태를 살피고, 무...","안상식 씨는 나훈아의 구슬픈 가락을 거실에 작게 틀어놓고 멍하니 생각에 잠기거나, ...",안상식 씨는 고등학교 동창들과 함께 창덕궁의 후원을 천천히 걸으며 왕실의 기록을 살...,안상식 씨는 일주일에 한 번 가족들과 함께 동네 단골 한우집에서 육즙 가득한 갈비를...,안상식 씨는 말수 적은 할아버지로서 손주들에게 잔소리 대신 정직하게 살아온 뒷모습을...,"안상식 씨는 목동에서 평생을 성실함과 규칙으로 무장해 온, 꼼꼼한 가계부와 나훈아의...","양천구 목동의 오래된 아파트 단지에서 수십 년째 거주하며, 정해진 시간에 일어나 집...","가계부의 십 원 단위까지 정확하게 맞추고, 집안의 모든 중요 서류를 날짜와 항목별로...",...,배우자있음,비현역,배우자와 거주,아파트,고등학교,해당없음,무직,서울-양천구,서울,대한민국


In [2]:
# 조건: 충남, 24세, 남자, 직업은 무직(=학생일 가능성), 텍스트에 '대학생'/'전공'/'학과' 언급
candidate = df[
    (df['province'] == '충청남') &
    (df['age'] == 24) &
    (df['sex'] == '남자') &
    (df['occupation'] == '무직') &
    (df['persona'].str.contains('대학생|학과|전공|캠퍼스', na=False))
]

print(f"조건에 맞는 후보 수: {len(candidate)}명")
candidate[['uuid', 'age', 'sex', 'occupation', 'district', 'education_level',
           'persona', 'career_goals_and_ambitions']].to_string()

조건에 맞는 후보 수: 17명


'                                    uuid  age sex occupation     district education_level                                                                                           persona                                                                                                                                                     career_goals_and_ambitions\n10697   7d91910328614dd3818e324f98c5e97d   24  남자         무직      충청남-아산시      2~3년제 전문대학       윤승훈 씨는 전기공학 전공의 실무 능력을 갖춘 24세 청년으로, 아산에서 안정적인 직장 생활을 꿈꾸며 느긋한 일상과 야구 응원을 통해 삶의 활력을 찾는 인물입니다.                                                                                    아산 산업단지 내 중견기업의 공정 관리직으로 입사하여 규칙적인 출퇴근 시간을 확보하고, 매달 꼬박꼬박 들어오는 월급으로 적금을 붓는 안정적인 삶을 꾸리고자 합니다.\n106490  59d82db9af7748ab86b2e399941633ee   24  남자         무직      충청남-부여군             대학원              하동민 씨는 정밀 설계 능력을 갖춘 공학 전공자이자, 부여의 정적인 삶과 현실적인 취업 고민 사이에서 갈등하며 느슨한 일상을 보내는 20대 청년입니다.                                          전공을 살려 지역 내 대규모 제조 공장이나 공기업의 기술직으로 들어가 안정적

In [3]:
step1 = df[(df['province'] == '충청남') & (df['age'] == 24) & (df['sex'] == '남자')]

display_cols = ['uuid', 'occupation', 'district', 'education_level',
                 'marital_status', 'family_type', 'housing_type']

pretty_view = step1[display_cols].reset_index(drop=True)
pretty_view.index = pretty_view.index + 1  # 1번부터 시작
pretty_view

,uuid,occupation,district,education_level,marital_status,family_type,housing_type
1,02a76e12eff647478f409598818f4f15,시설 경비원,충청남-태안군,4년제 대학교,미혼,어머니와 동거,단독주택
2,6e48b4a3861f486fb6225000bf752af1,전자계측 제어 기술자 및 연구원,충청남-아산시,4년제 대학교,미혼,혼자 거주,비주거용 건물 내 주택
3,fbedbd48a49a4c9d9ae84ec16f28dd21,그 외 제품 디자이너,충청남-논산시,4년제 대학교,미혼,기타2세대,아파트
4,7d91910328614dd3818e324f98c5e97d,무직,충청남-아산시,2~3년제 전문대학,미혼,혼자 거주,아파트
5,0ca8781f9d6f4e139c476014370f8c74,시스템 소프트웨어 설계 및 분석가,충청남-천안시 동남구,4년제 대학교,미혼,부모와 동거,단독주택
...,...,...,...,...,...,...,...
262,8e2c4dd643184b35867668c9f27920b9,경영 기획 사무원,충청남-계룡시,4년제 대학교,미혼,부모·조모와 동거,아파트
263,6f4904187b6846788b6918816efb205d,경리 사무원,충청남-서산시,4년제 대학교,미혼,아버지와 동거,아파트
264,777c84a1d5c34acb892ad6007212b1cb,컴퓨터 운영 관리자,충청남-천안시 서북구,4년제 대학교,미혼,부모와 동거,다세대주택
265,1f90e009abd24949aa6d5ac1f1aa27ec,무직,충청남-서산시,고등학교,미혼,부모와 동거,다세대주택


In [4]:
import re

step1 = df[(df['district'] == '충청남-계룡시') & (df['age'] == 24) & (df['sex'] == '남자')]

# persona 맨 앞의 "이름 씨는" 패턴에서 이름만 추출
def extract_name(text):
    match = re.match(r'^([가-힣]{2,4})\s*씨는', str(text))
    return match.group(1) if match else None

step1 = step1.copy()
step1['name'] = step1['persona'].apply(extract_name)

display_cols = ['name', 'uuid', 'occupation', 'district', 'education_level', 'persona']
step1[display_cols]

,name,uuid,occupation,district,education_level,persona
31513,오준서,1c48b1e108f04df38ad54716bd7eea07,공연·영화 및 음반 기획자,충청남-계룡시,4년제 대학교,"오준서 씨는 계룡시에서 홀로 거주하며 소극장의 꿈을 키우는 24세의 공연 기획자로,..."
222162,한준성,61d4caf729954a6a875dbd35b5a91632,무직,충청남-계룡시,4년제 대학교,"한준성 씨는 넉살 좋은 성격과 기계공학 전공 역량을 갖췄지만, 꼼꼼함이 부족하고 건..."
332415,유윤환,7de71c3df42640648f30649a1252fecf,무직,충청남-계룡시,4년제 대학교,"유윤환 씨는 법학 전공자의 꼼꼼한 분석력과 무직 취준생의 나른한 일상이 공존하는, ..."
482215,이호재,d35232ee4d2f443b8a92c6633bbe5d42,정보 시스템 운영자,충청남-계룡시,4년제 대학교,"이호재 씨는 계룡시의 안정적인 환경에서 성장해 교육기관 시스템 운영자로 일하며, 철..."
645779,이하람,0ef297eda179431eb31b3df6b5c25efa,무직,충청남-계룡시,2~3년제 전문대학,"이하람 씨는 감각적인 영상 편집 능력을 갖춘 취업 준비생으로, 계룡의 정적인 풍경 ..."
780603,김대호,febc51195dd6439da301c8320a856906,무직,충청남-계룡시,4년제 대학교,"김대호 씨는 계룡시에 거주하며 판교의 클라우드 엔지니어를 꿈꾸는 ICT 전공자로, ..."
868767,곽진우,5fa471582bc345098cb8a692de0dc743,시멘트 및 석회 제조 관련 조작원,충청남-계룡시,고등학교,곽진우 씨는 계룡시 시멘트 공장의 유능한 조작원이자 조부모님을 극진히 모시는 손자로...
936056,임한별,ddf5d9381e684c5c9fedbd516d48ff73,육군 병사,충청남-계룡시,2~3년제 전문대학,"임한별 씨는 계룡시에서 자란 성실한 육군 병사로, 꼼꼼한 업무 능력과 가족에 대한 ..."
971779,유성하,8e2c4dd643184b35867668c9f27920b9,경영 기획 사무원,충청남-계룡시,4년제 대학교,유성하 씨는 계룡시에서 농업 행정을 담당하며 가족을 아끼는 다정한 성격의 20대 청...


100만 명의 가상 주민들 중에서 제 최애는 오준서씨입니다.

나이도 저와 같고 사는 지역도 같아서, 그냥 훑어보다가 자연스럽게 눈에 들어왔습니다.

 대학생이라는 것까지 겹쳐서 여러모로 공통점이 많았던 것 같습니다.

그런데 직업을 보니 저와는 전공 방향이 완전히 다른 공연·영화 및 음반 기획자를 꿈꾸고 있더라고요.

 나이나 지역, 신분은 다 같은데 진로만 이렇게 다르다는 게 오히려 재밌게 느껴져서 더 기억에 남았습니다.

In [ ]:
import json
from google.colab import files

path = "/content/EDA1 (3).ipynb"

with open(path, "r", encoding="utf-8") as f:
    nb = json.load(f)

if "widgets" in nb.get("metadata", {}):
    del nb["metadata"]["widgets"]

with open(path, "w", encoding="utf-8") as f:
    json.dump(nb, f, ensure_ascii=False, indent=1)

with open(path, "r", encoding="utf-8") as f:
    check = json.load(f)
print("widgets 남음?:", "widgets" in check.get("metadata", {}))
print("셀 개수:", len(check.get("cells", [])))

files.download(path)